# Flywheel DICOM Download — Single Subject

This notebook downloads raw DICOM data for a single participant from UPenn Flywheel to the fMRI server.

For downloading multiple subjects at once, see `flywheel_download_batch.ipynb`.

**Note:** You do not have to use this script. As long as your DICOMs are organized in the expected directory structure under `/fmriDataRaw/fmri_data_raw/{PROJECT}/{SUBJECT}/`, the rest of the pipeline will work regardless of how files were transferred.

---

### References
- [Flywheel Python SDK Docs](https://api-docs.flywheel.io/latest/index.html)
- [Flywheel Docs](https://docs.flywheel.io/)

## 1. Imports

In [ ]:
import flywheel
import tarfile
import zipfile
import os
import configparser

## 2. API Key Setup

Your Flywheel API key is stored in a local config file (`~/configs/config.ini`) that is excluded from version control.

- Find your API key at: https://upenn.flywheel.io/#/profile
- **Never share or commit your API key**

Run the cell below **only on first use**, or if your API key has changed. This will write the key to your config file. **Do not** save your API key within this notebook.

In [ ]:
# --- FIRST-TIME SETUP ONLY ---
# Set your API key and run this cell once to save it to your config file.
# After that, skip this cell and use the 'Read API Key' cell below.

user = os.environ["USER"]
home_dir = f"/home/{user}"

your_api_key = "upenn.flywheel.io:YOUR_API_KEY_HERE"  # <-- replace with your key

config = configparser.ConfigParser()
config['UPENN-FLYWHEEL'] = {'apikey': your_api_key}

os.makedirs(f"{home_dir}/configs", exist_ok=True)
with open(f"{home_dir}/configs/config.ini", 'w') as config_file:
    config.write(config_file)

print(f"API key saved to {home_dir}/configs/config.ini")

### Read API Key from Config

In [ ]:
# Read API key from config file
user = os.environ["USER"]
home_dir = f"/home/{user}"

config = configparser.ConfigParser()
config.read(f"{home_dir}/configs/config.ini")

if not config.has_option('UPENN-FLYWHEEL', 'apikey'):
    raise ValueError("API key not found. Run the setup cell above first.")

api_key = config['UPENN-FLYWHEEL']['apikey']
print("API key loaded successfully.")

## 3. Connect to Flywheel & Verify Access

In [ ]:
# Initialize Flywheel client and confirm authentication
fw = flywheel.Client(api_key)

current_user = fw.get_current_user()
print(f"Connected to Flywheel as: {current_user.firstname} {current_user.lastname} ({current_user.email})")

## 4. Set Project & Subject Parameters

Edit the variables in this cell before running the download.

Your Flywheel data is organized as: **Group → Project → Subject → Session**  
You can find these values in the Flywheel URL, e.g.:
`https://upenn.flywheel.io/#/projects/{project_id}/sessions/{session_id}`

In [ ]:
# ── Flywheel identifiers ───────────────────────────────────────────────────────
group_label   = "your_group"        # Flywheel group (e.g. lab name)
project_label = "your_project"      # Flywheel project label
session_label = "CAMRIS^Falk"       # Session label as it appears on Flywheel

# Subject ID as it appears on Flywheel
subject_id_flywheel = "sub-001"

# ── Local output identifiers ───────────────────────────────────────────────────
# Project folder name on the fMRI server (under /fmriDataRaw/fmri_data_raw/)
out_project = "your_local_project"

# Subject ID for server storage (can match Flywheel ID or follow your own convention)
subject_id_local = "sub-001"

# ── Derived paths ─────────────────────────────────────────────────────────────
# Root output directory for raw DICOMs on the server
outpath = f"/fmriDataRaw/fmri_data_raw/{out_project}"

# Final destination for this subject's DICOMs
output_dicom_dir = os.path.join(outpath, subject_id_local)

print(f"Flywheel path : {group_label}/{project_label}/{subject_id_flywheel}/{session_label}")
print(f"Local output  : {output_dicom_dir}")

## 5. Verify Output Directory

In [ ]:
# Create the output directory if it does not already exist
if not os.path.exists(output_dicom_dir):
    os.makedirs(output_dicom_dir)
    print(f"Created output directory: {output_dicom_dir}")
else:
    print(f"Output directory already exists: {output_dicom_dir}")

# List current contents (should be empty on first run)
print("\nContents:", os.listdir(output_dicom_dir))

## 6. Look Up the Flywheel Session

In [ ]:
# Look up the session object on Flywheel using the group/project/subject path
# The session label is not used in the lookup — Flywheel matches on subject
lookup_string = f"{group_label}/{project_label}/{subject_id_flywheel}"
print(f"Looking up: {lookup_string}")

session = fw.lookup(lookup_string)
print("Session found:")
print(session)

## 7. Download Session Tarball

This downloads the full session as a `.tar` archive to a local `working_data/` directory.  
Files are typically ~1 GB. The archive is a temporary staging area and can be deleted after extraction.

In [ ]:
# Create a temporary working directory for the tarball
os.makedirs("working_data", exist_ok=True)

tar_path = f"./working_data/{subject_id_local}.tar"
print(f"Downloading session to: {tar_path}")

fw.download_tar(session, tar_path)

print("Download complete.")

## 8. Extract DICOMs from Tarball

The tarball contains one or more `dicom.zip` files (one per scan series).  
This cell extracts all DICOM files into the subject's output directory.

In [ ]:
# Open the tarball and extract any members that contain DICOM data
with open(tar_path, 'rb') as f:
    tar_data = tarfile.open(fileobj=f, mode='r:')
    
    extracted_count = 0
    for member in tar_data.getmembers():
        
        if 'dicom.zip' in member.name:  # only extract DICOM zip files
            print(f"  Extracting: {member.name}")
            
            tfile = tar_data.extractfile(member.name)
            dicom_zip = zipfile.ZipFile(tfile, mode='r')
            dicom_zip.extractall(output_dicom_dir)
            extracted_count += 1
    
    tar_data.close()

print(f"\nExtracted {extracted_count} DICOM archive(s) to: {output_dicom_dir}")

## 9. Verify Output

In [ ]:
# List the extracted files/directories to confirm the download was successful
contents = os.listdir(output_dicom_dir)
print(f"Files in {output_dicom_dir} ({len(contents)} items):")
for item in sorted(contents):
    print(f"  {item}")

## 10. Clean Up (Optional)

The `.tar` file in `working_data/` can be safely deleted once you have verified the DICOMs extracted correctly. Flywheel retains the original data and you can re-download at any time.

> ⚠️ **Double-check the path before running.** This will recursively delete everything in the working_data directory.

In [ ]:
# Uncomment and run to delete the tarball after confirming successful extraction

# import shutil
# shutil.rmtree('./working_data/')
# print('working_data/ deleted.')